# 02 - smolagents

Second in the **Agent Frameworks** series. Same use case as
[01 - AIMU.ipynb](01%20-%20AIMU.ipynb); only the framework changes. See [README.md](../README.md) for
the shared design.

## What smolagents brings

[smolagents](https://github.com/huggingface/smolagents) (Hugging Face) is a minimal agent library.
Tools are plain functions with the `@tool` decorator; the agent runs a ReAct loop. We use
`ToolCallingAgent` (explicit JSON tool calls) for parity with the other notebooks; smolagents'
flagship `CodeAgent` (which writes Python to call tools) is noted at the end.

smolagents has **no built-in document store**, so we use a ChromaDB collection with local Ollama
embeddings, wired in through tools — the idiomatic external-store pattern for smolagents. It also
has no built-in evaluator loop, so we express the generate→evaluate→revise loop as a short Python
loop with a critic model call.

## 01 - Setup

```bash
ollama pull qwen3.6:27b
ollama pull nomic-embed-text
```

In [ ]:
import os

import dotenv

import chromadb
from chromadb.utils import embedding_functions
from smolagents import LiteLLMModel, ToolCallingAgent, tool

from genscai import research, paths

dotenv.load_dotenv(paths.root / ".env")

MODEL = f"ollama_chat/{os.environ.get('GENSCAI_AGENT_MODEL', 'qwen3.6:27b')}"
OLLAMA_URL = "http://localhost:11434"

RESEARCH_QUESTION = (
    "What interventions and control strategies have recent preprints proposed or evaluated for "
    "dengue outbreaks, and what evidence do they report?"
)

model = LiteLLMModel(model_id=MODEL)

## 02 - The shared research tools

Every notebook in the series uses the same `genscai.research` search/fetch functions (medRxiv +
bioRxiv via the Europe PMC keyword API).

In [ ]:
for h in research.search_medrxiv("dengue vaccination", max_results=3):
    print(h["date"], "-", h["title"][:80])

## 03 - The document store and agent tools

A ChromaDB collection (local Ollama embeddings) is smolagents' document store here. The agent reads
each abstract and calls `save_relevant_paper` **only** for matches, so the store stays curated.

In [ ]:
ef = embedding_functions.OllamaEmbeddingFunction(url=OLLAMA_URL, model_name="nomic-embed-text")
chroma = chromadb.PersistentClient(path=str(paths.output / "agent_frameworks" / "smolagents_store"))
collection = chroma.get_or_create_collection("papers", embedding_function=ef)

_seen: dict[str, dict] = {}


@tool
def search_preprints(query: str) -> str:
    """Search medRxiv and bioRxiv preprints. Returns each hit's DOI, title, date, and abstract.

    Args:
        query: A free-text search query; always include the key topic term (e.g. 'dengue').
    """
    results = research.search_medrxiv(query, max_results=5) + research.search_biorxiv(query, max_results=3)
    if not results:
        return "No results found."
    blocks = []
    for article in results:
        _seen[article["doi"]] = article
        blocks.append(
            f"DOI: {article['doi']}\nTitle: {article['title']}\nDate: {article['date']}\n"
            f"Abstract: {(article['abstract'] or '')[:600]}"
        )
    return "\n\n".join(blocks)


@tool
def save_relevant_paper(doi: str) -> str:
    """Save a paper you have confirmed is relevant to the research question, identified by its DOI.

    Args:
        doi: The DOI of a paper returned by a previous search.
    """
    article = _seen.get(doi)
    if not article:
        return f"Unknown DOI {doi}. Search for it first so its details are available."
    document = f"{article['title']}\n\n{article['abstract']}"
    collection.upsert(
        ids=[doi],
        documents=[document],
        metadatas=[{"title": article["title"], "authors": article["authors"] or "", "date": article["date"] or "", "url": article["url"] or ""}],
    )
    return f"Saved: {article['title']}"


@tool
def read_saved_papers() -> str:
    """Read every paper saved in the local document store, to ground your synthesis."""
    data = collection.get()
    if not data["ids"]:
        return "No papers saved yet."
    blocks = []
    for doi, meta, doc in zip(data["ids"], data["metadatas"], data["documents"]):
        blocks.append(f"# {meta['title']}\nDOI: {doi} | Date: {meta['date']}\nURL: {meta['url']}\n\n{doc}")
    return "\n\n---\n\n".join(blocks)

## 04 - The researcher agent

`ToolCallingAgent` runs the ReAct loop. smolagents prints a rich trace of each step (tool calls and
observations) as it runs. `reset=True` (the default for `run`) isolates each run; the ChromaDB store
persists across runs.

In [ ]:
SYSTEM = (
    "You are a research assistant building a cited literature review grounded in a curated corpus. "
    "Workflow: (1) call read_saved_papers to see what is curated; (2) use search_preprints, always "
    "including the key topic term (e.g. 'dengue') in queries; (3) for each candidate, if its abstract "
    "is plausibly relevant call save_relevant_paper(doi), erring toward saving; (4) before synthesizing "
    "you MUST call read_saved_papers and base the synthesis only on those papers, citing each by title "
    "and DOI. Never claim no papers were found if the store is non-empty. Be concise."
)

agent = ToolCallingAgent(
    tools=[search_preprints, save_relevant_paper, read_saved_papers],
    model=model,
    instructions=SYSTEM,
    max_steps=12,
)

synthesis = agent.run(RESEARCH_QUESTION)
print(synthesis)

## 05 - The self-correcting loop

smolagents has no built-in evaluator-optimizer, so we write the loop ourselves: a critic model call
returns `PASS` or revision feedback, and we re-run the agent with that feedback. The curated store
persists, so each revision builds on it.

In [ ]:
def critique(question, synthesis):
    prompt = (
        "Review this literature synthesis. Reply with exactly PASS if it answers the question, cites "
        "specific papers (title + DOI), and reports evidence. Otherwise reply REVISE: <specific gap>.\n\n"
        f"Question: {question}\n\nSynthesis:\n{synthesis}"
    )
    response = model.generate([{"role": "user", "content": [{"type": "text", "text": prompt}]}])
    return response.content


MAX_ROUNDS = 2
for round_num in range(MAX_ROUNDS):
    feedback = critique(RESEARCH_QUESTION, synthesis)
    print(f"--- critic (round {round_num + 1}): {feedback[:120]}")
    if "PASS" in feedback:
        break
    synthesis = agent.run(f"Revise your synthesis using this feedback:\n{feedback}\n\nQuestion: {RESEARCH_QUESTION}")

print("\n=== FINAL SYNTHESIS ===\n")
print(synthesis)

## 06 - Inspect the curated corpus

In [ ]:
data = collection.get()
print(f"{len(data['ids'])} papers saved:")
for meta in data["metadatas"]:
    print(" -", meta["title"][:80])

## 07 - smolagents notes

- **Tools**: `@tool` on a plain function; the docstring (with an `Args:` section) becomes the schema.
  smolagents inspects source, so tools must be defined in real cells/modules.
- **Document store**: none built in; we used a ChromaDB collection via tools. This is the typical
  smolagents pattern (the framework focuses on the agent loop, not storage).
- **The feedback loop**: hand-written. smolagents gives you the agent loop; orchestration like
  generate→evaluate→revise is yours to compose.
- **CodeAgent**: smolagents' signature agent writes Python code to call tools instead of emitting
  JSON tool calls. Swap `ToolCallingAgent` for `CodeAgent` to try it; it can chain tool calls in one
  step but is harder to constrain for a gated workflow like this one.
- **Local models**: via `LiteLLMModel("ollama_chat/<model>")` (LiteLLM's Ollama backend).